# MultiModalSCVI sweep — benchmark

Aggregates the cached sweep runs (`cache/sweep/<run_id>/`) trained by `sweep/train_sweep.py`.
CPU-only: loads cached latents + PPC metrics, no model reload / no GPU.
Benchmarks: UMAPs, PPC reconstruction, Moran's I autocorrelation histograms, scib metrics.

Grid: spatial(top500, ct100) × n_latent(10,20,30) × n_layers(2,3) × batch_mask(ff,ft) × experts(poe,moe) = 48 runs.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from itertools import product
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import pyplot as plt

from PixelGen.scvi_utils import pca_neighbors_umap
from PixelGen.utils import plot_composite_ppc
from PixelGen.metrics import distr_autocorrelation_in_latent

sc.set_figure_params(figsize=(5, 3), frameon=False)

NEW_DATA    = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
CACHE       = NEW_DATA / 'cache'
SWEEP_CACHE = CACHE / 'sweep'

ABUNDANCE_LAYER = 'arcsinh'
SPATIAL_GT      = 'spatial_asinh5_top500var'   # shared spatial ground-truth for autocorr
BATCH_KEY       = 'cell_system'
BIO_KEY         = 'cell_type_annot'

# Reconstruct the grid run_ids (matches train_sweep.make_run_id).
GRID = dict(spatial=['top500', 'ct100'], n_latent=[10, 20, 30], n_layers=[2, 3],
            batch_mask=['ff', 'ft'], experts=['poe', 'moe'])
run_ids = [f'sp-{sp}__nl{nl}__L{L}__bm-{bm}__{ex}'
           for sp, nl, L, bm, ex in product(*GRID.values())]
print(f'{len(run_ids)} grid configs')

In [ ]:
adata = sc.read_h5ad(CACHE / 'adata_cytovi_annotated_compat.h5ad')

# Attach each finished run's joint latent to obsm; track which runs are available.
available = []
for rid in run_ids:
    f = SWEEP_CACHE / rid / 'latents.npz'
    if not (SWEEP_CACHE / rid / 'DONE').exists() or not f.exists():
        continue
    adata.obsm[f'z__{rid}'] = np.load(f)['z_joint']
    available.append(rid)

latent_keys = [f'z__{rid}' for rid in available]
print(f'available: {len(available)}/{len(run_ids)}')
missing = [r for r in run_ids if r not in available]
if missing:
    print('missing:', *missing, sep='\n  ')

## UMAPs
48 is a lot of output — edit `RUNS_TO_PLOT` to subset.

In [ ]:
RUNS_TO_PLOT = available  # e.g. available[:8] or [r for r in available if 'poe' in r]
for rid in RUNS_TO_PLOT:
    pca_neighbors_umap(
        adata, latent_name=f'z__{rid}',
        umap_pl_kwargs=dict(color=[BIO_KEY, BATCH_KEY], frameon=False, ncols=2),
        umap_title=rid,
    )
    plt.show()

## PPC — reconstruction metrics
Each run reconstructs its own inputs. Abundance (`arcsinh`) is identical across runs; spatial ground-truth differs by spatial axis (top500 vs ct100), so the Spatial panel measures self-reconstruction, not a head-to-head.

In [ ]:
metrics_df = pd.concat(
    [pd.read_parquet(SWEEP_CACHE / rid / 'ppc_metrics.parquet') for rid in available],
    ignore_index=True,
)
plot_composite_ppc('Abundance', metrics_df, 'cornflowerblue', available); plt.show()
plot_composite_ppc('Spatial',   metrics_df, 'lightgreen',     available); plt.show()

## Moran's I autocorrelation
Per run, build a kNN graph on the joint latent and compute Moran's I on abundance + spatial features. Higher = smoother across neighbours. Spatial rep is the shared `spatial_asinh5_top500var` for all runs.

In [ ]:
autocorr_abundance = distr_autocorrelation_in_latent(
    adata, latent_keys=latent_keys, names=available,
    rep_key=ABUNDANCE_LAYER, pca_kwargs={'n_comps': 15},
)
autocorr_spatial = distr_autocorrelation_in_latent(
    adata, latent_keys=latent_keys, names=available,
    rep_key=SPATIAL_GT, pca_kwargs={'n_comps': 15},
)
print('abundance:', autocorr_abundance.shape, '| spatial:', autocorr_spatial.shape)

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=autocorr_spatial, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[0], legend=False)
axes[0].set_title(f"Spatial Moran's I  (rep={SPATIAL_GT})")
axes[0].set_xlabel("Moran's I")

sns.histplot(data=autocorr_abundance, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[1], legend=False)
axes[1].set_title(f"Abundance Moran's I  (rep={ABUNDANCE_LAYER})")
axes[1].set_xlabel("Moran's I")
plt.tight_layout(); plt.show()

In [ ]:
# Mean / median Moran's I per run, ranked.
rows = []
for label, df in [('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]:
    for name in available:
        s = df[df['latent'] == name]['morans']
        rows.append({'Feature type': label, 'Model': name,
                     'Mean': s.mean(), 'Median': s.median(), 'Std': s.std()})
summary_df = pd.DataFrame(rows)
summary_df.pivot(index='Model', columns='Feature type', values=['Mean', 'Median']).sort_values(('Mean', 'Spatial'), ascending=False)

## scib metrics
bio = `cell_type_annot`, batch = `cell_system`, across all run joint latents.

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

bm = Benchmarker(
    adata,
    batch_key=BATCH_KEY,
    label_key=BIO_KEY,
    embedding_obsm_keys=latent_keys,
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
)
bm.benchmark()
bm.plot_results_table(min_max_scale=False)